## 1\. Import libraries

In [ ]:
# ---- Step 1: Import libraries ----
import pandas as pd
import os
import glob
import re

from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime, date
from zoneinfo import ZoneInfo

# \(A\) QC FILE VALIDATION

## 2\. Load QC file \(raw, no changes yet\)

In [ ]:
# ---- Step 2: Load QC file (raw) ----
# Auto-detects whichever format is actually sitting in the folder — .csv or .xlsx —
# so switching source formats in the future doesn't require editing this cell again.
qc_folder = "from_qcAnalyst"
qc_files_csv = glob.glob(os.path.join(qc_folder, "qc_report_*.csv"))
qc_files_xlsx = glob.glob(os.path.join(qc_folder, "qc_report_*.xlsx"))
qc_files = qc_files_csv + qc_files_xlsx

if len(qc_files) == 0:
    print("No QC file found. Check the folder name or file name format.")
elif len(qc_files) > 1:
    print("More than one QC file found. Expected only one.")
    for f in qc_files:
        print(" -", f)
else:
    qc_path = qc_files[0]
    ext = os.path.splitext(qc_path)[1].lower()

    if ext == ".csv":
        qc_df = pd.read_csv(qc_path)
    elif ext == ".xlsx":
        qc_df = pd.read_excel(qc_path)
    else:
        raise ValueError(f"Unrecognized QC file extension: '{ext}'. Expected .csv or .xlsx.")

    print(f"QC file loaded: {qc_path}")
    print(f"Rows: {len(qc_df)}")
    print(f"Columns: {list(qc_df.columns)}")

## 2A\. Normalize all string columns

\(uppercase, remove whitespace\) — must run first, so validation only catches real format issues, not casing/spacing typos

In [ ]:
# this block converts all col w/ string to a uniform format, all caps, no white spaces.
# this is to account and correct typos (eg white spaces, sometimes in lower caps etc)

def clean_column(column):
    """
    Remove all whitespace and convert text to uppercase.
    Only transforms non-null values — real blanks/NaN are never passed through
    .astype(str), so they can never become the literal text "NAN"/"NONE".
    """
    return column.where(
        column.isna(),
        column.astype(str).str.replace(r"\s+", "", regex=True).str.upper()
    )

str_col_to_clean = ['lot_number', 'product_code',
                'customer', 'original_lot', 'evaluated_by',
                'updated_by', 'updated_on', 'qc_type', 'formula_id',
                'operator', 'supervisor', 'bag_no', 'internal_lot']

for each_col in str_col_to_clean:
    qc_df[each_col] = clean_column(qc_df[each_col])

In [ ]:
print (qc_df.head())

## 2B\. Check ID for missing/gaps

> check missing prod ID in orig db

In [ ]:
# ---- Step 2B: Validate QC file - Check IDs for missing/gaps ----
id_col = "id"

qc_ids = qc_df[id_col].dropna().astype(int)
id_min, id_max = qc_ids.min(), qc_ids.max()
full_range = set(range(id_min, id_max + 1))
missing_ids = sorted(full_range - set(qc_ids))
duplicate_ids = qc_ids[qc_ids.duplicated()].tolist()

print("=" * 60)
print("QC FILE - ID VALIDATION")
print("=" * 60)
print(f"ID range: {id_min} to {id_max}")
print(f"Total rows: {len(qc_ids)}")

if missing_ids:
    print(f"FLAGGED: {len(missing_ids)} missing ID(s) found.")
    # pd.DataFrame({"missing_id": missing_ids}).to_csv("flagged_missing_ids.csv", index=False)
    # print("Full list saved to flagged_missing_ids.csv for review.")
    # print(f"Missing ID sample: {missing_ids[:20]}")
else:
    print("No missing IDs found. Proceeding.")

if duplicate_ids:
    print(f"FLAGGED: {len(duplicate_ids)} duplicate ID(s) found.")
    print(f"Duplicate ID sample: {duplicate_ids[:20]}")
else:
    print("No duplicate IDs found.")

# ---- Step 2A (addendum): Split missing IDs by Aug 1, 2025 cutoff ----
"""
Dev noted missing IDs should not occur after Aug 1, 2025 — 
Implementation of new QC program any post-cutoff gap means
the record was excluded from the report export, not missing from the database.
It means that the record have value FALSE in is_active column in database.
"""

cutoff_id = 6493  # confirmed: first ID with encoded_on >= Aug 1, 2025

missing_before_cutoff = [i for i in missing_ids if i < cutoff_id]
missing_after_cutoff = [i for i in missing_ids if i >= cutoff_id]

print(f"\nMissing IDs before Aug 1, 2025 (id < {cutoff_id}): {len(missing_before_cutoff)}")
print(f"Missing IDs on/after Aug 1, 2025 (id >= {cutoff_id}): {len(missing_after_cutoff)}")

if missing_after_cutoff:
    print(f"FLAGGED: {len(missing_after_cutoff)} missing ID(s) found on & after the new program deployment.")
    # print(f"Sample: {missing_after_cutoff[:20]}")
else:
    print("No missing IDs on/after cutoff — consistent with dev's note.")

## 2C\. Validate Product Codes

In [ ]:
# ---- Step 2C: Validate QC file - Check product_code value if matched the gathered format ----
import re

# Confirmed formats (Convo A):
#   XX-X00000X  (5-digit, with client prefix)   e.g. DP-G13757E
#   XX-X0000X   (4-digit, with client prefix)   e.g. same structure, fewer digits
#   XX00000X or XX0000X (no prefix, plain code) e.g. RA16826E or VA4086E
PRODUCT_CODE_PATTERN = re.compile(
    r"^([A-Za-z]{2}-[A-Za-z]\d{4,5}[A-Za-z]|[A-Za-z]{2}\d{4,5}[A-Za-z])$"
)

def validate_product_code(value):
    if pd.isna(value):
        return False, "Blank/NaN product_code"
    value = str(value).strip()
    if not PRODUCT_CODE_PATTERN.match(value):
        return False, "Does not match confirmed format"
    return True, "OK"

print("=" * 60)
print("QC FILE - PRODUCT_CODE VALIDATION")
print("=" * 60)

product_code_issues = []

for idx, row in qc_df.iterrows():
    is_valid, reason = validate_product_code(row["product_code"])
    if not is_valid:
        product_code_issues.append({
            "row_index": idx,
            "id": int(row["id"]),
            "product_code": row["product_code"],
            "reason": reason
        })

if product_code_issues:
    print(f"FLAGGED: {len(product_code_issues)} row(s) with unexpected product_code format.")
    if len(product_code_issues) <= 20:
        for issue in product_code_issues:
            print(f"  ID {issue['id']}: '{issue['product_code']}'")
    else:
        pd.DataFrame(product_code_issues).to_csv("flagged_product_code.csv", index=False)
        print("Full list saved to flagged_product_code.csv for review.")
else:
    print("All product_code values passed validation.")

TEST BLOCK: TRY APPLYING CORRECTIONS\.

In [ ]:
# ---- Step 2C (continued): Apply confirmed corrections and re-validate product_code ----

# New format confirmed with Jam:
#   XX0000X-X   (plain code + dash + 1 letter suffix) e.g. WA3827E-I — rare, confirmed with Ms. Jam
PRODUCT_CODE_PATTERN_UPDATED = re.compile(
    r"^([A-Za-z]{2}-[A-Za-z]\d{4,5}[A-Za-z]|[A-Za-z]{2}\d{4,5}[A-Za-z]|[A-Za-z]{2}\d{4,5}[A-Za-z]-[A-Za-z])$"
)

def validate_product_code_updated(value):
    if pd.isna(value):
        return False, "Blank/NaN product_code"
    value = str(value).strip()
    if not PRODUCT_CODE_PATTERN_UPDATED.match(value):
        return False, "Does not match confirmed format"
    return True, "OK"

# ---- Confirmed corrections from Ms. Jam for the 10 flagged rows ----
# Kept separate from the raw file — original value is preserved, correction is
# stored temporarily and only applied to the merged output later.
PRODUCT_CODE_CORRECTIONS = {
    13180: "DP-G10045E",
    9187: "WA15190E",
    9179: "WA15190E",   # confirmed typo, same as WA15190E
    5928: "WA15190E",
    3844: "DV-I16082E",
    3739: "WA15123E",
    3046: "GA1225E",
    3045: "GA1225E",
    2188: "RA12739E",
    # 12286 (WA3827E-I) is not a typo — it's a newly confirmed valid format, no correction needed
}

qc_df["product_code_corrected"] = qc_df.apply(
    lambda row: PRODUCT_CODE_CORRECTIONS.get(int(row["id"]), row["product_code"]),
    axis=1
)

print("=" * 60)
print("QC FILE - PRODUCT_CODE VALIDATION (after corrections)")
print("=" * 60)

product_code_issues_updated = []

for idx, row in qc_df.iterrows():
    is_valid, reason = validate_product_code_updated(row["product_code_corrected"])
    if not is_valid:
        product_code_issues_updated.append({
            "row_index": idx,
            "id": int(row["id"]),
            "product_code": row["product_code"],
            "product_code_corrected": row["product_code_corrected"],
            "reason": reason
        })

if product_code_issues_updated:
    print(f"FLAGGED: {len(product_code_issues_updated)} row(s) with unexpected product_code format.")
    if len(product_code_issues_updated) <= 20:
        for issue in product_code_issues_updated:
            print(f"  ID {issue['id']}: '{issue['product_code']}'")
    else:
        pd.DataFrame(product_code_issues_updated).to_csv("flagged_product_code.csv", index=False)
        print("Full list saved to flagged_product_code.csv for review.")
else:
    print("All product_code values passed validation after corrections.")

LENGTH CHECK AFTER APPLYING TEST CORRECTIONS IN PRODUCT CODE:

In [ ]:
def is_covered_format(value):
    value = str(value).strip()

    if re.match(r"^[A-Za-z]{2}-[A-Za-z]\d{4,5}[A-Za-z]$", value):
        return True
    if re.match(r"^[A-Za-z]{2}\d{4,5}[A-Za-z]-[A-Za-z]$", value):
        return True
    if re.match(r"^[A-Za-z]{2}\d{4,5}[A-Za-z]$", value):
        return True

    return False

qc_df["id"] = qc_df["id"].astype(int)
qc_df["product_code_is_covered"] = qc_df["product_code_corrected"].apply(is_covered_format)

uncovered = qc_df[qc_df["product_code_is_covered"] == False]

print(f"Total rows: {len(qc_df)}")
print(f"Covered by known format: {(qc_df['product_code_is_covered'] == True).sum()}")
print(f"NOT covered by any known format: {len(uncovered)}")

if len(uncovered) > 0:
    print("\nUncovered product_code values:")
    print(uncovered[["id", "product_code_corrected"]].to_string(index=False))
else:
    print("\nAll product_code values are covered by a known format.")

## 2D\. Validate Lot Number

Confirm all real formats present, narrowed down step by step: single lot → range → parenthesis \(bag\-in\-parens / internal\-lot\-in\-parens\) →        whatever remains is flagged

In [ ]:
# check lot format

# put all lot data in a list
lot_number_list = qc_df['lot_number'].tolist()

### \- 2D Validation Check 1: Exclude valid lot numbers in the list\.

In [ ]:
# CHECK THE POSSIBLE VALID FORMATS of the lot number column

#(1) remove the ones where format is SSSSNN or SSSSN as it is valid 
pattern = re.compile(r"^\d{4}[A-Za-z]{1,2}$")
invalid_lot_number_list = [item for item in lot_number_list if not pattern.match(str(item))]


# NOTE
# lot_number_list len = 12,339
# invalid_lot_number_list len  = 3,879


### \- 2D Validation Check 2: List valid lot number ranges\.

In [ ]:
#(2) remove those w/ this format SSSSNN-SSSSNN and SSSSN-SSSSN where , 
# all 3 conditions below must be met 
#    a) all number must have "- " in b/w
#    b) b4 and after must be in the format listed above and same on each side
#    c) first number < 2nd number 
# as they are all VALID 



'''
for (a) and (b)
pattern is : 
    There is exactly one -
    Left side is either: NNNNS (4 digits + 1 letter), or  NNNNSS (4 digits + 2 letters)
    Right side is the same format as the left side (either both 1 letter or both 2 letters).
'''
pattern = re.compile(
    r'^(?:\d{4}[A-Za-z]-\d{4}[A-Za-z]|\d{4}[A-Za-z]{2}-\d{4}[A-Za-z]{2})$'
)

# temp_list are the VALID lot number 
temp_list = [item for item in invalid_lot_number_list if pattern.match(str(item))]




In [ ]:
print (temp_list)
print (len(temp_list))

### \- 2D Validation Check 3: Confirm valid lot number range order

In [ ]:
#then (c)... must be in consecutive order or 1st<2nd , eg 1000AB-1200AB or 1000AA-1000AB,  
# if not then remove them as they are invalid


def remove_invalid_ranges(my_list):
    pattern = re.compile(r"^(\d{4})([A-Za-z]{1,2})\s*-\s*(\d{4})([A-Za-z]{1,2})$")

    cleaned_list = []

    for item in my_list:
        match = pattern.match(str(item).strip())

        # If format does not match, keep it
        if not match:
            cleaned_list.append(item)
            continue

        nnnn = int(match.group(1))
        left_letters = match.group(2).upper()

        NNNN = int(match.group(3))
        right_letters = match.group(4).upper()

        # If left and right letter length are not the same, keep it
        if len(left_letters) != len(right_letters):
            cleaned_list.append(item)
            continue

        remove_item = False

        # Case 1: 4 digits + 1 letter
        if len(left_letters) == 1:
            s = left_letters
            S = right_letters

            if s == S:
                if NNNN <= nnnn:
                    remove_item = True
            else:
                if s > S:
                    remove_item = True

        # Case 2: 4 digits + 2 letters
        elif len(left_letters) == 2:
            ss = left_letters
            SS = right_letters

            t = ss[-1]
            T = SS[-1]

            if ss == SS:
                if NNNN <= nnnn:
                    remove_item = True
            else:
                if t > T:
                    remove_item = True

        if not remove_item:
            cleaned_list.append(item)

    return cleaned_list



temp_list = remove_invalid_ranges(temp_list)

# seems nothing was remove and ranges are valid

# remove items in temp_list from invalid_lot_number_list as temp_list are all valid lots
invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in temp_list
]

# invalid_Lot_number_list len now at 1000

In [ ]:
print (invalid_lot_number_list)
print (len(invalid_lot_number_list))

### \- 2D Validation Check 4: Remove valid parenthesis\-format bag numbers \(single bag\)

In [ ]:
# Format: 1234AB(#) or 1234A(#) — valid base lot, single bag number in parentheses
# e.g. 7200AM(10), 6921AM(4), 3067AM(2)

pattern = re.compile(
    r'^\d{4}[A-Za-z]{1,2}\(\d+\)$'
)

# temp_list_bag_single are the VALID entries for this pass
temp_list_bag_single = [item for item in invalid_lot_number_list if pattern.match(str(item).strip())]

print(temp_list_bag_single)
print(len(temp_list_bag_single))


In [ ]:
# Remove these confirmed-valid entries from invalid_lot_number_list
invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in temp_list_bag_single
]

print(f"Remaining invalid after Check 4: {len(invalid_lot_number_list)}")

### \- 2D Validation Check 5: Remove valid parenthesis\-format bag ranges

In [ ]:
# ---- Step 2D Pass 5:  ----
# Format: 1234AB(#a-#b) where a < b — valid base lot, bag range in parentheses
# e.g. 7200AM(1-9), 7461AM(35-37), 7594AM(38-39)

pattern = re.compile(
    r'^\d{4}[A-Za-z]{1,2}\((\d+)-(\d+)\)$'
)

temp_list_bag_range = []

for item in invalid_lot_number_list:
    match = pattern.match(str(item).strip())
    if match:
        a, b = int(match.group(1)), int(match.group(2))
        if a < b:
            temp_list_bag_range.append(item)

print(temp_list_bag_range)
print(len(temp_list_bag_range))

In [ ]:
# Remove these confirmed-valid entries from invalid_lot_number_list
invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in temp_list_bag_range
]

print(f"Remaining invalid after Check 5: {len(invalid_lot_number_list)}")

### \- 2D Validation Check 6: Remove valid parenthesis\-format internal lot numbers

In [ ]:
# ---- Step 2D Pass 6:  ----
# Format: 1234AB(5678CD) — valid base lot, real internal lot (letters + numbers) in parentheses
# e.g. 6124AM(6127AM), 6240AM(6249AM), 2904AM(2914AM)

pattern = re.compile(
    r'^\d{4}[A-Za-z]{1,2}\((\d{4}[A-Za-z]{1,2})\)$'
)

temp_list_internal_lot = [item for item in invalid_lot_number_list if pattern.match(str(item).strip())]

print(temp_list_internal_lot)
print(len(temp_list_internal_lot))

# Remove these confirmed-valid entries from invalid_lot_number_list
invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in temp_list_internal_lot
]

print(f"Remaining invalid after Check 6: {len(invalid_lot_number_list)}")

In [ ]:
print(invalid_lot_number_list)
print(len(invalid_lot_number_list))

### \- 2D Validation Check 7: Check invalid Lot Numbers

In [ ]:
print(invalid_lot_number_list)
print(len(invalid_lot_number_list))

TEST BLOCK: TRY APPLYING CORRECTIONS FOR TYPOS\.

In [ ]:
# ---- Step 2D: Apply confirmed corrections for lot_number anomalies ----
# Kept separate from the raw file — original value is preserved, correction is
# stored temporarily and only applied to the merged output later.

LOT_NUMBER_CORRECTIONS = {
    "33478AO"         : "3478AO",
    "9345AM-9348AM`"  : "9345AM-9348AM",
    "8670AM-7671AM"   : "8670AM-8671AM",
    "4550-4551AM"     : "4550AM-4551AM",
    "0764AM-0365AM"   : "0364AM-0365AM",
    "86342AL(8344AL)" : "8342AL(8344AL)",
    "6965AL`"         : "6965AL",
    "5466AL(5473AL("  : "5466AL(5473AL)",
    "4506AL--4522AL"  : "4506AL-4522AL",
    "4034AAL-4039AL"  : "4034AL-4039AL",
}

qc_df["lot_number_corrected"] = qc_df["lot_number"].apply(
    lambda val: LOT_NUMBER_CORRECTIONS.get(str(val).strip(), val)
)

corrected_count = (qc_df["lot_number"] != qc_df["lot_number_corrected"]).sum()
print(f"Applied {corrected_count} lot_number correction(s).")
print(qc_df.loc[qc_df["lot_number"] != qc_df["lot_number_corrected"],
                 ["id", "lot_number", "lot_number_corrected"]])

In [ ]:
invalid_lot_number_list = [
    item for item in invalid_lot_number_list
    if str(item).strip() not in LOT_NUMBER_CORRECTIONS
]

print(invalid_lot_number_list)
print(len(invalid_lot_number_list))

# ---- Normalize lot_number_corrected / internal_lot bag+OS variants ----
# Covers: lot(n), lot (n), lot(n-n), lot (n-n), lot(n) OS, lot OS(n)/lot OS (n),
# lot(n-n) OS, lot OS(n-n)/lot OS (n-n)
# Canonical output: LOT(n) or LOT(n-n), with " OS" appended at the end if present.

LOT_BAG_OS_PATTERN = re.compile(
    r'^(\d{4}[A-Za-z]{1,2})'          # base lot: 0000XX or 0000X
    r'(?:\s*OS)?'                      # optional OS before paren
    r'\s*\((\d{1,3}(?:-\d{1,3})?)\)'   # (n) or (n-n)
    r'(?:\s*OS)?$',                    # optional OS after paren
    re.IGNORECASE
)

# Reversed variant: bag number in parentheses comes BEFORE the lot number,
# e.g. "(1-7)3854AL", "(1-7) 3854AL", "(7)3854AL" — confirmed a real QC data-entry
# pattern, not a typo. Normalized to the same canonical LOT(bag) form as the
# standard variant so downstream classification/matching treats it identically.
PREFIX_BAG_LOT_PATTERN = re.compile(
    r'^\((\d{1,3}(?:-\d{1,3})?)\)'    # (n) or (n-n) leading
    r'\s*'
    r'(\d{4}[A-Za-z]{1,2})'           # base lot: 0000XX or 0000X
    r'(?:\s*OS)?$',                    # optional trailing OS
    re.IGNORECASE
)

def normalize_lot_bag_os(value):
    if pd.isna(value):
        return value
    value = str(value).strip().upper()

    m = LOT_BAG_OS_PATTERN.match(value)
    if m:
        base_lot, bag_part = m.group(1), m.group(2)
        has_os = "OS" in value
        result = f"{base_lot}({bag_part})"
        if has_os:
            result += " OS"
        return result

    m_prefix = PREFIX_BAG_LOT_PATTERN.match(value)
    if m_prefix:
        bag_part, base_lot = m_prefix.group(1), m_prefix.group(2)
        has_os = "OS" in value
        result = f"{base_lot}({bag_part})"
        if has_os:
            result += " OS"
        return result

    return value  # doesn't match any known variant — leave as-is for existing validation to flag

qc_df["lot_number_corrected"] = qc_df["lot_number_corrected"].apply(normalize_lot_bag_os)

if "internal_lot" in qc_df.columns:
    qc_df["internal_lot"] = qc_df["internal_lot"].apply(normalize_lot_bag_os)

normalized_count = qc_df["lot_number_corrected"].apply(
    lambda v: bool(LOT_BAG_OS_PATTERN.match(str(v))) if pd.notna(v) else False
).sum()
print(f"\nlot_number_corrected values matching lot(bag)/OS pattern: {normalized_count}")

### \- 2D Validation Check 8: Remove valid lot number range \+ parenthesis\-format bag ranges

In [ ]:
# ---- Step 2D: Identify valid Lot Ranged + Parenthesised Ranged Bag (equal count) ----
# Format: 0000AA-0001AA(5-6) — valid ONLY if the count of lots in the range
# matches the count of bags in the range. Each lot pairs to one bag in sequential order.
# ex: 0000AA(5), 0001AA(6)

pattern = re.compile(
    r'^(\d{4})([A-Za-z]{1,2})-(\d{4})([A-Za-z]{1,2})\((\d+)-(\d+)\)$'
)

resolved_equal_originals = []

for item in invalid_lot_number_list:
    match = pattern.match(str(item).strip())
    if not match:
        continue

    start_num, start_letters, end_num, end_letters, bag_start, bag_end = match.groups()

    # Letters on both sides of the lot range must match (same series)
    if start_letters.upper() != end_letters.upper():
        continue

    lot_count = int(end_num) - int(start_num) + 1
    bag_count = int(bag_end) - int(bag_start) + 1

    # Valid only if lot count matches bag count, and both are in ascending order
    if lot_count == bag_count and int(end_num) > int(start_num) and int(bag_end) > int(bag_start):
        resolved_equal_originals.append(item)

print(resolved_equal_originals)
print(len(resolved_equal_originals))

TEST BLOCK: TRY APPLYING CORRECTIONS FOR 3 RANGE \+ BAG RANGE \(ONE LOT = ONE BAG\)

In [ ]:
# ---- Step 2D: Split each confirmed equal-count combo entry into individual lot(bag) pairs ----

temp_list_lot_range_bag_range = []

for item in resolved_equal_originals:
    match = pattern.match(str(item).strip())
    start_num, start_letters, end_num, end_letters, bag_start, bag_end = match.groups()
    num_len = len(start_num)

    lot_numbers = [f"{str(n).zfill(num_len)}{start_letters.upper()}" for n in range(int(start_num), int(end_num) + 1)]
    bag_numbers = list(range(int(bag_start), int(bag_end) + 1))

    for lot, bag in zip(lot_numbers, bag_numbers):
        temp_list_lot_range_bag_range.append(f"{lot}({bag})")

print(temp_list_lot_range_bag_range)
print(len(temp_list_lot_range_bag_range))

invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in resolved_equal_originals
]

print(f"Remaining invalid after Check 8: {len(invalid_lot_number_list)}")

In [ ]:
invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in temp_list_lot_range_bag_range
]

print(invalid_lot_number_list)
print(len(invalid_lot_number_list))

TEST BLOCK: TRY APPLYING CORRECTIONS FOR REMAINING INVALID LOT NUMBER

In [ ]:
# ---- Manual corrections for uneven Lot Range + Bag Range mismatches ----
# These 3 entries have a lot range count that does NOT match the bag range count,
# so they can't auto-split evenly. Corrected manually and split per-lot as confirmed.

# (3) has a raw data error in the bag range itself — fix that first before splitting.
RAW_VALUE_CORRECTIONS = {
    "7786AL-7788AL(17-24)": "7786AL-7788AL(15-24)",
}

# Confirmed manual split — each original range entry maps to its corrected individual lot+bag rows.
UNEVEN_RANGE_BAG_SPLITS = {
    "7190AM-7192AM(15-19)": ["7190AM(15-16)", "7191AM(17-18)", "7192AM(19)"],
    "7839AL-7840AL(9-11)": ["7839AL(9-10)", "7840AL(11)"],
    "7786AL-7788AL(15-24)": ["7786AL(15-16)", "7787AL(17-20)", "7788AL(21-24)"],
}

temp_list_uneven_split = []
resolved_originals = []

for original_value in list(invalid_lot_number_list):
    corrected_raw = RAW_VALUE_CORRECTIONS.get(original_value, original_value)

    if corrected_raw in UNEVEN_RANGE_BAG_SPLITS:
        temp_list_uneven_split.extend(UNEVEN_RANGE_BAG_SPLITS[corrected_raw])
        resolved_originals.append(original_value)

print(temp_list_uneven_split)
print(len(temp_list_uneven_split))

In [ ]:
# Remove these resolved entries from invalid_lot_number_list
invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in resolved_originals
]

print(invalid_lot_number_list)
print(len(invalid_lot_number_list))

## 2E\. Classify QC lot\_number initial formats

In [ ]:
def classify_qc_lot(value):
    value = str(value).strip()

    if value == "" or value.lower() == "nan":
        return "blank"
    if "(" in value and ")" in value:
        return "parenthesis"
    if "-" in value:
        return "range"
    return "single"

qc_df["lot_format"] = qc_df["lot_number_corrected"].apply(classify_qc_lot)

print("Count of each lot_number format:")
print(qc_df["lot_format"].value_counts())
print(f"\nTotal classified rows: {qc_df['lot_format'].value_counts().sum()}")
print(f"Total raw QC file rows (after dropping blank spacer rows): {len(qc_df)}")

## 2F\. Expand range\-format lot numbers

In [ ]:
# ---- Step 2G: Expand range-based QC lot_number values into individual lots ----

# Lot number format 0000XX - Split into value (0000 and XX)
def split_lot_parts(lot): 
    match = re.match(r"^(\d+)([A-Za-z]+)$", lot.strip())
    if not match:
        return None, None, None
    number_str, letters = match.group(1), match.group(2)
    return number_str, letters, len(number_str)

# Get the next letter combo in the lot number string, like AM turning into AN.
def next_letters(letters): 
    letters = list(letters.upper())
    i = len(letters) - 1
    while i >= 0:
        if letters[i] != 'Z':
            letters[i] = chr(ord(letters[i]) + 1)
            break
        else:
            letters[i] = 'A'
            i -= 1
    return "".join(letters)

# Split the range: eg 6222AM-6224AM -> 6222AM, 6223AM, 6224AM
def expand_range(start_lot, end_lot): 
    start_num_str, start_letters, num_len = split_lot_parts(start_lot)
    end_num_str, end_letters, _ = split_lot_parts(end_lot)

    if start_num_str is None or end_num_str is None:
        return None

    result = []
    current_num = int(start_num_str)
    current_letters = start_letters
    max_value = 10 ** num_len - 1

    safety_counter = 0
    max_iterations = 5000

    while True:
        current_lot = f"{str(current_num).zfill(num_len)}{current_letters}"
        result.append(current_lot)

        if current_lot == end_lot.strip().upper():
            break

        current_num += 1
        if current_num > max_value:
            current_num = 1
            current_letters = next_letters(current_letters)

        safety_counter += 1
        if safety_counter > max_iterations:
            print(f"WARNING: range {start_lot}-{end_lot} exceeded safety limit, stopped expanding.")
            break

    return result

# ---- Apply range expansion to all range-format rows ----
range_rows = qc_df[qc_df["lot_format"] == "range"].copy()

expanded_records = []
failed_ranges = []

for idx, row in range_rows.iterrows():
    parts = row["lot_number_corrected"].split("-")
    if len(parts) != 2:
        failed_ranges.append(row["lot_number_corrected"])
        continue

    start_lot, end_lot = parts[0].strip(), parts[1].strip()
    expanded_lots = expand_range(start_lot, end_lot)

    if expanded_lots is None:
        failed_ranges.append(row["lot_number_corrected"])
        continue

    for lot in expanded_lots:
        new_row = row.copy()
        new_row["expanded_lot_number"] = lot
        expanded_records.append(new_row)

expanded_df = pd.DataFrame(expanded_records)

print(f"Total range rows: {len(range_rows)}")
print(f"Total individual lots after expansion: {len(expanded_df)}")
print(f"Ranges that failed to parse: {len(failed_ranges)}")
if failed_ranges:
    # print("Failed range samples:", failed_ranges[:10])
    print("Failed range samples:", failed_ranges)

## 2G\. Lot number format breakdown

In [ ]:
# ---- Step 2H Part 1: Build ALL_COMBO_CORRECTIONS from earlier resolved lists ----
# Reuses what earlier passes already computed, instead of hardcoding a second copy.

ALL_COMBO_CORRECTIONS = {}

# From the uneven manual splits
for original_value in resolved_originals:  # e.g. 7190AM-7192AM(15-19)
    corrected_raw = RAW_VALUE_CORRECTIONS.get(original_value, original_value)
    ALL_COMBO_CORRECTIONS[original_value] = UNEVEN_RANGE_BAG_SPLITS[corrected_raw]

# From the equal-count auto splits
for original_value in resolved_equal_originals:  # e.g. 8761AL-8762AL(39-40)
    match = pattern.match(original_value)
    start_num, start_letters, end_num, _, bag_start, bag_end = match.groups()
    num_len = len(start_num)
    lot_numbers = [f"{str(n).zfill(num_len)}{start_letters.upper()}" for n in range(int(start_num), int(end_num) + 1)]
    bag_numbers = list(range(int(bag_start), int(bag_end) + 1))
    ALL_COMBO_CORRECTIONS[original_value] = [f"{lot}({bag})" for lot, bag in zip(lot_numbers, bag_numbers)]

print(ALL_COMBO_CORRECTIONS)
print(f"Total combo entries built: {len(ALL_COMBO_CORRECTIONS)}")

In [ ]:
# ---- Step 2H Part 2: Expand combo rows into qc_df, then classify all lot_number formats ----

rows_to_expand = qc_df[qc_df["lot_number_corrected"].isin(ALL_COMBO_CORRECTIONS.keys())]
expanded_combo_records = []

for idx, row in rows_to_expand.iterrows():
    for new_lot in ALL_COMBO_CORRECTIONS[row["lot_number_corrected"]]:
        new_row = row.copy()
        new_row["lot_number_corrected"] = new_lot
        expanded_combo_records.append(new_row)

qc_df = qc_df[~qc_df["lot_number_corrected"].isin(ALL_COMBO_CORRECTIONS.keys())]
qc_df = pd.concat([qc_df, pd.DataFrame(expanded_combo_records)], ignore_index=True)

print(f"Expanded {len(rows_to_expand)} combo row(s) into {len(expanded_combo_records)} individual row(s).")
print(f"qc_df total rows now: {len(qc_df)}")

# ---- Classify every lot_number format ----
BASE_LOT = r"\d{4}[A-Za-z]{1,2}"

PATTERNS = {
    "single":             re.compile(rf"^{BASE_LOT}$"),
    "range":              re.compile(rf"^{BASE_LOT}-{BASE_LOT}$"),
    "bag_single_paren":   re.compile(rf"^{BASE_LOT}\(\d+\)$"),
    "bag_range_paren":    re.compile(rf"^{BASE_LOT}\(\d+-\d+\)$"),
    "internal_lot_paren": re.compile(rf"^{BASE_LOT}\({BASE_LOT}\)$"),
}

format_buckets = {name: [] for name in PATTERNS}
unclassified = []

for idx, row in qc_df.iterrows():
    value = str(row["lot_number_corrected"]).strip()
    entry = {"id": int(row["id"]), "product_code": row["product_code_corrected"], "lot_number": value}

    matched_any = False
    for name, pattern_re in PATTERNS.items():
        if pattern_re.match(value):
            format_buckets[name].append(entry)
            matched_any = True
            break

    if not matched_any:
        unclassified.append(entry)

print("=" * 60)

for name, entries in format_buckets.items():
    print(f"{name}: {len(entries)} row(s)")

print(f"\nUnclassified (no known format matched): {len(unclassified)} row(s)")
if unclassified:
    for r in unclassified:
        print(f"  ID {r['id']}: product_code='{r['product_code']}', lot_number='{r['lot_number']}'")
else:
    print("All lot_number values are now covered by a known format.")

In [ ]:
# ---- Step 2H (addendum): Explain the row count change ----
# qc_df grew from 12,339 to 12,347 rows after this step.
# This is expected — not a data error. Here's why:

print("=" * 60)
print("ROW COUNT CHANGE EXPLANATION")
print("=" * 60)
print(f"Rows before this step  : 12339")
print(f"Rows after this step   : {len(qc_df)}")
print(f"Difference             : {len(qc_df) - 12339}")

print("\nReason: 6 QC rows had a 'lot range + bag range' combo format")
print("(e.g. 8761AL-8762AL(39-40)), which is not a single real lot — it actually")
print("represents multiple individual lots, each with its own bag number.")
print("This step expanded each of those 6 rows into several individual rows,")
print("one per real lot(bag) pair, so each one can be matched correctly later.")

print("\nBreakdown of the expansion:")
for original, expanded_list in ALL_COMBO_CORRECTIONS.items():
    print(f"  '{original}'  ->  {len(expanded_list)} row(s): {expanded_list}")

total_expanded = sum(len(v) for v in ALL_COMBO_CORRECTIONS.values())
print(f"\n6 original rows expanded into {total_expanded} individual rows.")
print(f"Net change: {total_expanded} - 6 = {total_expanded - 6} additional rows.")

## 2H\. Clean bag number in QC file

In [ ]:
# ---- Create bag_no_corrected in qc_df (must run before Block 5) ----
MONTH_MAP = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

def fix_bag_no(value):
    if pd.isna(value) or str(value).strip() == '':
        return None
    value = str(value).strip()

    # Detect parenthesis wrapping: (n) or (n-n) — strip for processing, re-wrap after
    paren_match = re.match(r'^\((.+)\)$', value)
    is_paren = bool(paren_match)
    inner = paren_match.group(1).strip() if is_paren else value

    def replace_month(match):
        return str(MONTH_MAP[match.group(0).lower()])
    fixed = re.sub(r'(?i)(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)', replace_month, inner)

    parts = fixed.split('-')
    if len(parts) == 2:
        try:
            a, b = int(parts[0].strip()), int(parts[1].strip())
            result = f'{min(a,b)}-{max(a,b)}'
            return f'({result})' if is_paren else result
        except:
            pass
    else:
        try:
            n = int(fixed.strip())
            result = str(n)
            return f'({result})' if is_paren else result
        except:
            pass

    return f'({fixed})' if is_paren else fixed

qc_df["bag_no_corrected"] = qc_df["bag_no"].apply(fix_bag_no)
print("bag_no_corrected created.")

changed = qc_df[qc_df["bag_no"].astype(str).str.strip() != qc_df["bag_no_corrected"].astype(str).str.strip()]
print(len(changed))

# ---- Validate bag_no_corrected against approved formats: n, n-n, (n), (n-n) ----
BAG_FORMAT_PATTERN = re.compile(r'^(\(\d{1,3}(-\d{1,3})?\)|\d{1,3}(-\d{1,3})?)$')

def is_valid_bag_format(value):
    if value is None or pd.isna(value) or str(value).strip() == '':
        return True  # blank is not a format issue
    return bool(BAG_FORMAT_PATTERN.match(str(value).strip()))

qc_df["bag_format_valid"] = qc_df["bag_no_corrected"].apply(is_valid_bag_format)

invalid_bag_format = qc_df[qc_df["bag_format_valid"] == False]
print(f"\nRows with bag_no_corrected NOT matching approved format: {len(invalid_bag_format)}")
if len(invalid_bag_format):
    print(invalid_bag_format[["id", "product_code", "lot_number", "bag_no", "bag_no_corrected"]].to_string(index=False))

CLEAN BAG NUMBER IN REMARKS COLUMN

In [ ]:
# List all bags in remarks column
BAG_WORD_PATTERN = re.compile(r'\bbag\b', re.IGNORECASE)

has_bag_in_remarks = qc_df["remarks"].apply(
    lambda x: bool(BAG_WORD_PATTERN.search(str(x))) if pd.notna(x) else False
)

subset = qc_df.loc[has_bag_in_remarks, ["id", "product_code", "lot_number", "remarks"]]
print(f"Records with 'bag' mentioned in remarks: {len(subset)}\n")
print(subset.to_string(index=False, max_colwidth=100))

In [ ]:
def normalize_bag_spacing(remarks_value):
    if pd.isna(remarks_value):
        return remarks_value
    text = str(remarks_value)
    text = re.sub(r'(?i)\bmega\s+bag\b', 'megabag', text)  # "mega bag", "mega  bag" -> "megabag"
    text = re.sub(r'(?i)\bbag\s*#\s*', 'bag#', text)        # "bag # 1", "bag  #1" -> "bag#1"
    text = re.sub(r'(?i)#\s+', '#', text)                    # any remaining "# 1" -> "#1"
    return text

qc_df["remarks_normalized"] = qc_df["remarks"].apply(normalize_bag_spacing)

changed_spacing = qc_df[qc_df["remarks"].astype(str) != qc_df["remarks_normalized"].astype(str)]
print(f"Remarks with spacing normalized: {len(changed_spacing)}\n")
print(changed_spacing[["id", "remarks", "remarks_normalized"]].to_string(index=False, max_colwidth=50))

In [ ]:
BAG_REMARKS_PATTERN = re.compile(
    r'\bbag\b\s*#?\s*(\d{1,3}(?:\s*-\s*\d{1,3})?)',
    re.IGNORECASE
)

def extract_bag_from_remarks(remarks_value):
    if pd.isna(remarks_value):
        return None
    match = BAG_REMARKS_PATTERN.search(str(remarks_value))
    if match:
        return match.group(1).replace(' ', '')
    return None

needs_fallback = qc_df["bag_no_corrected"].isna() | (qc_df["bag_no_corrected"].astype(str).str.strip() == '')
qc_df.loc[needs_fallback, "bag_no_corrected"] = qc_df.loc[needs_fallback, "remarks_normalized"].apply(extract_bag_from_remarks)

recovered = qc_df.loc[needs_fallback & qc_df["bag_no_corrected"].notna()]
print(f"Bag numbers recovered from remarks: {len(recovered)}\n")
print(recovered[["id", "product_code", "lot_number", "remarks_normalized", "bag_no_corrected"]].to_string(index=False, max_colwidth=80))

In [ ]:
def fix_mega_typos(remarks_value):
    if pd.isna(remarks_value):
        return remarks_value
    text = str(remarks_value)
    text = re.sub(r'(?i)\bmaga\s+bag\b', 'megabag', text)  # "maga bag" -> "megabag"
    text = re.sub(r'(?i)\bmeg\s+bag\b', 'megabag', text)   # "meg bag" -> "megabag"
    return text

qc_df["remarks_normalized"] = qc_df["remarks_normalized"].apply(fix_mega_typos)

# Re-run the extraction now that the typos are fixed
qc_df.loc[needs_fallback, "bag_no_corrected"] = qc_df.loc[needs_fallback, "remarks_normalized"].apply(extract_bag_from_remarks)

recovered = qc_df.loc[needs_fallback & qc_df["bag_no_corrected"].notna()]
print(f"Bag numbers recovered from remarks (after typo fix): {len(recovered)}\n")
print(recovered[["id", "product_code", "lot_number", "remarks_normalized", "bag_no_corrected"]].to_string(index=False, max_colwidth=50))

In [ ]:
VALID_BAG_FORMAT = re.compile(r'^\d{1,3}(-\d{1,3})?$')

verify_df = qc_df.loc[needs_fallback & qc_df["bag_no_corrected"].notna()].copy()
verify_df["format_valid"] = verify_df["bag_no_corrected"].astype(str).str.match(VALID_BAG_FORMAT)
verify_df["status"] = verify_df["format_valid"].map({True: "OK", False: "FAILED"})

print(f"Total bag numbers recovered from remarks: {len(verify_df)}")
print(f"Passed format validation: {verify_df['format_valid'].sum()}")
print(f"Failed format validation: {(~verify_df['format_valid']).sum()}\n")

print(verify_df[["id", "product_code", "lot_number", "remarks_normalized", "bag_no_corrected", "status"]].to_string(index=False, max_colwidth=50))

# \(B\) SPECTRO FILES VALIDATION

## 3\. Load SPECTRO file \(raw, no changes yet\)

In [ ]:
# ---- Step 3: Load Spectro files (raw, no changes yet) ----
spectro_folder = "from_spectro"
spectro_files = glob.glob(os.path.join(spectro_folder, "*.xlsx"))

spectro_data = {}       # product_code -> dataframe
spectro_filepaths = {}  # product_code -> original full filepath (for traceability downstream)

print("Spectro files found:")
for f in spectro_files:
    filename = os.path.basename(f)
    name_no_ext = os.path.splitext(filename)[0]  # strip extension first
    product_code = name_no_ext.split(" ")[0]     # text before first space, extension-safe
    df = pd.read_excel(f)
    spectro_data[product_code] = df
    spectro_filepaths[product_code] = f
    print(f" - {product_code}: {len(df)} rows, file = '{filename}'")

print(f"\nTotal Spectro files loaded: {len(spectro_data)}")

In [ ]:
# ---- Step 2H (new): Drop fully-blank separator rows from raw Spectro data ----
# These are structural export artifacts (blank row before a new STD/reference block),
# not real data — confirmed against BA12615E and BA12861E raw files. A row with
# every column NaN can never be a real reading, so drop it before any normalization
# touches the Name column (this also prevents the NaN->"NAN" string bug downstream).

for code, df in spectro_data.items():
    before = len(df)
    df = df.dropna(how="all").reset_index(drop=True)
    dropped = before - len(df)
    spectro_data[code] = df
    if dropped:
        print(f"{code}: dropped {dropped} fully-blank row(s)")

## 3A\. Normalize Name column \(Column B\) before validation

In [ ]:
# ---- Step 3A: Normalize Column B (Name) per Spectro file ----
for code, df in spectro_data.items():
    name_col = "Name"  # Column B

    # Preserve the untouched raw value before any changes — same pattern as QC's raw column
    df["Name_raw"] = df[name_col]

    df[name_col] = (
        df[name_col]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)  # collapse multiple spaces to one (not remove entirely — spacing is meaningful here)
        .str.strip()
        .str.upper()
    )

print("Column B normalized (uppercase, trimmed, single spaces) for all Spectro files.")
print("Original raw values preserved in 'Name_raw' column.")


In [ ]:
# ---- Step 3A (addendum): Print normalized Column B values per Spectro file ----
for code, df in spectro_data.items():
    name_col = "Name"
    print(f"\n--- {code} ---")
    print(f"Total rows: {len(df)}")
    print(df[name_col].tolist())

In [ ]:
# ---- Step 3A (addendum): Strip "junk text + product code" prefix from Column B ----
# Confirmed pattern: some Spectro entries prepend free text (e.g. customer/collector name)
# followed by the file's own product code, before the real lot+bag+OS value.
#   e.g. "EVEREST KA2871E 9891AM" (file KA2871E.xlsx) -> real value is "9891AM"
# Only strips when the file's OWN product code appears as a whole word inside the value,
# followed by more text — untouched otherwise, so normal rows are never affected.

def strip_product_code_prefix(value, code):
    value_str = str(value).strip()
    escaped_code = re.escape(str(code).strip().upper())
    pattern = re.compile(rf"^.*?\b{escaped_code}\b\s+(.+)$", re.IGNORECASE)
    m = pattern.match(value_str)
    if m:
        return m.group(1).strip()
    return value_str

for code, df in spectro_data.items():
    before_values = df["Name"].copy()
    df["Name"] = df["Name"].apply(lambda v: strip_product_code_prefix(v, code))

    changed = df[before_values != df["Name"]]
    if len(changed):
        print(f"{code}: stripped prefix on {len(changed)} row(s)")
        for idx in changed.index:
            print(f"  '{before_values[idx]}' -> '{df.loc[idx, 'Name']}'")

In [ ]:
# ---- Step 3A (addendum 2): Convert "bag/bag/lot" slash-format into standard Name format ----
# Confirmed pattern: some Spectro entries record bag+lot as "start/end/lot4digit" using
# slashes instead of the normal Name format, always implying an "AM" lot letter suffix
# (this only occurs when the lot has an "AM" suffix that Excel stripped off).
#   e.g. "11/12/7461" -> start=11, end=12, lot=7461 -> "7461AM 11-12" (bag range)
#        "1/1/7461"   -> start == end == 1        -> "7461AM 1"      (single bag)
#        "12/11/7461" -> start > end               -> INVALID, left unconverted so it
#                        fails downstream format validation and reports NO MATCH FOUND.
# Converts to the standard "LOT BAG" text so no other downstream cell needs to change.

SLASH_BAG_LOT_PATTERN = re.compile(r"^(\d{1,3})/(\d{1,3})/(\d{4})$")

slash_invalid_flags = []  # (product_code, row_index, original_value) for start > end

# Matches the stringified form Step 3A leaves behind, e.g. "8817-07-08 00:00:00"
# or "8817-07-08" (year can be >4 digits since Excel let '7/8/8817' roll into one).
EXCEL_DATE_STRING_PATTERN = re.compile(r"^(\d{4,})-(\d{2})-(\d{2})(?:\s+00:00:00)?$")

def reverse_excel_date_slash(value):
    """
    Excel auto-converts text like '7/8/8817' into an actual date (month=7, day=8,
    year=8817) on load. By the time this runs, Step 3A has already stringified it
    to 'YYYY-MM-DD 00:00:00' (uppercased). This reconstructs the original 'M/D/Y'
    slash text from that string before the slash-format regex tries to match it.
    """
    if isinstance(value, (pd.Timestamp, datetime, date)):
        return f"{value.month}/{value.day}/{value.year}"

    value_str = str(value).strip()
    m = EXCEL_DATE_STRING_PATTERN.match(value_str)
    if m:
        year, month, day = m.groups()
        return f"{int(month)}/{int(day)}/{int(year)}"

    return value

def convert_slash_bag_lot(value, code=None, idx=None):
    value = reverse_excel_date_slash(value)
    value_str = str(value).strip()
    m = SLASH_BAG_LOT_PATTERN.match(value_str)
    if not m:
        return value_str

    bag_start_str, bag_end_str, lot_num = m.groups()
    bag_start, bag_end = int(bag_start_str), int(bag_end_str)
    lot_out = f"{lot_num}AM"

    if bag_start > bag_end:
        # Invalid range — do not convert, leave as-is so it fails validation later.
        slash_invalid_flags.append((code, idx, value_str))
        return value_str

    if bag_start == bag_end:
        bag_out = bag_start_str          # single bag number
    else:
        bag_out = f"{bag_start_str}-{bag_end_str}"   # genuine bag range

    return f"{lot_out} {bag_out}"

for code, df in spectro_data.items():
    before_values = df["Name"].copy()
    df["Name"] = [convert_slash_bag_lot(v, code, idx) for idx, v in df["Name"].items()]

    changed = df[before_values != df["Name"]]
    if len(changed):
        print(f"{code}: converted slash-format bag/lot on {len(changed)} row(s)")
        for idx in changed.index:
            print(f"  '{before_values[idx]}' -> '{df.loc[idx, 'Name']}'")

if slash_invalid_flags:
    print(f"\nFLAGGED: {len(slash_invalid_flags)} slash-format row(s) with start > end (invalid range):")
    for code, idx, val in slash_invalid_flags:
        print(f"  [{code}] Row {idx}: '{val}' — left unconverted, will report NO MATCH FOUND")


## 3B\. Clean reference rows

### \- 3B Validation Check 1: Identify reference rows

In [ ]:
# ---- Identify reference rows (STD, LIGHT, DARK, CMA, %) ----
def is_reference_row(value):
    if pd.isna(value):
        return True
    value = str(value).upper()
    return any(marker in value for marker in ["STD", "LIGHT", "DARK", "CMA", "%"])

for code, df in spectro_data.items():
    name_col = "Name"
    df["is_reference_row"] = df[name_col].apply(is_reference_row)

    ref_count = df["is_reference_row"].sum()
    print(f"{code}: {ref_count} reference row(s) identified out of {len(df)} total")

In [ ]:
for code, df in spectro_data.items():
    name_col = "Name"
    ref_rows = df[df["is_reference_row"] == True]

    print(f"\n--- {code} ---")
    print(f"Reference row(s) found: {len(ref_rows)}")
    print(ref_rows[name_col].tolist())

### \- 3B Re\-arrangement of lot numbers & product codes

In [ ]:
# Split reference rows into product_code and lot_number ----

REFERENCE_MARKER_PATTERN = re.compile(r"\b(STD|LIGHT|DARK)\b(\s+UPDATED\b)?", re.IGNORECASE)
MARKER_SHORT = {"STD": "STD", "LIGHT": "LT", "DARK": "DR"}

def clean_reference_lot_number(value, product_code):
    value = str(value).strip().upper()
    code_upper = str(product_code).strip().upper()

    if value.startswith(code_upper):
        value = value[len(code_upper):].strip()

    match = REFERENCE_MARKER_PATTERN.search(value)
    if match:
        marker = match.group(1).upper()
        has_updated = bool(match.group(2))
        remainder = (value[:match.start()] + value[match.end():]).strip()
        remainder = re.sub(r"\s+", " ", remainder)

        short = MARKER_SHORT[marker]
        prefix = f"{short} UPDATED" if has_updated else short
        value = f"{prefix} {remainder}".strip()

    return value


for code, df in spectro_data.items():
    ref_mask = df["is_reference_row"] == True
    name_col = "Name"

    # Create product_code column beside Name, blank for non-reference rows for now
    if "product_code" not in df.columns:
        df.insert(df.columns.get_loc(name_col) + 1, "product_code", None)

    for idx in df[ref_mask].index:
        df.loc[idx, "product_code"] = code  # the file's own product code
        df.loc[idx, name_col] = clean_reference_lot_number(df.loc[idx, name_col], code)

    # Rename Name column to "lot number"
    df.rename(columns={name_col: "lot number"}, inplace=True)

    print(f"\n--- {code} ---")
    print(df.loc[ref_mask, ["lot number", "product_code"]].to_string(index=False))

## 3C\. Validate Column B \(Name\)

### \- 3C Differentiate Lot Numbers to Reference Lot Numbers

In [ ]:
# ---- Print total row counts and reference row counts across all Spectro files ----

total_rows_all_files = 0
total_reference_rows_all_files = 0

for code, df in spectro_data.items():
    total_rows_all_files += len(df)
    total_reference_rows_all_files += df["is_reference_row"].sum()

print(f"Total rows across all Spectro files: {total_rows_all_files}")
print(f"Total reference rows across all Spectro files: {total_reference_rows_all_files}")
print(f"Total real (non-reference) rows: {total_rows_all_files - total_reference_rows_all_files}")

### \- 3C Identify valid and invalid lot number formats

In [ ]:
# put all lot data in a list
valid_spectro_names = []
invalid_spectro_names = []

In [ ]:
# ---- Step 3C Block 1: Identify valid vs invalid lot number formats (real rows only) ----
import re

BASE_LOT = r"\d{4}[A-Za-z]{1,2}"        # 0000A or 0000AA only
BAG_CORE = r"\d{1,3}(?:-\d{1,3})?"      # single/double/triple digit, or a ranged combo of the same
BAG_ANY  = rf"#?{BAG_CORE}"             # bag with an optional leading "#"
BAG_HASH = rf"#{BAG_CORE}"              # bag that REQUIRES a leading "#" (only used for no-space-OS forms)

SPECTRO_NAME_PATTERN = re.compile(
    rf"^{BASE_LOT}("
    rf"\sOS\s{BAG_ANY}"          # OS <space> bag, plain or #      e.g. "1234A OS 1", "5409AN OS #19"
    rf"|\sOS{BAG_HASH}"          # OS#bag, attached, # required     e.g. "5409AN OS#19"
    rf"|\s{BAG_ANY}\sOS"         # bag <space> OS, plain or #       e.g. "1234A 1 OS", "5409AN #19 OS"
    rf"|\s{BAG_HASH}OS"          # #bagOS, attached, # required      e.g. "5409AN #19OS"
    rf"|\s{BAG_ANY}"             # plain space-separated bag         e.g. "1234A 1", "5409AN #19"
    rf"|\sOS"                     # OS alone, no bag                  e.g. "1234A OS"
    rf"|\s?\({BAG_CORE}\)\s?OS"   # paren bag + OS, space optional both sides
                                    # e.g. "1234A(1)OS", "1234A (1) OS", "3672AL(35-36) OS"
    rf"|\s?\({BAG_CORE}\)"        # paren bag, optional leading space e.g. "1234A(1)", "1271AL (1)"
    rf")?$",
    re.IGNORECASE
)

# valid_spectro_names = []
# invalid_spectro_names = []

for code, df in spectro_data.items():
    real_rows = df[df["is_reference_row"] == False]

    for idx, row in real_rows.iterrows():
        value = row["lot number"]
        value_str = str(value).strip() if not pd.isna(value) else ""

        if not pd.isna(value) and SPECTRO_NAME_PATTERN.match(value_str):
            valid_spectro_names.append({"product_code": code, "row_index": idx, "value": value_str})
        else:
            invalid_spectro_names.append({"product_code": code, "row_index": idx, "value": value_str})

print("Valid lot number values:")
for entry in valid_spectro_names:
    print(f"  [{entry['product_code']}] Row {entry['row_index']}: '{entry['value']}'")

In [ ]:
print(f"Valid lot number format count: {len(valid_spectro_names)}")

### \- 3C Get invalid Lot Number format

In [ ]:
# ---- Step 3C Block 2: Print invalid lot number formats ----
if invalid_spectro_names:
    print(f"FLAGGED: {len(invalid_spectro_names)} row(s) with unexpected Column B format.")
    for issue in invalid_spectro_names:
        print(f"  [{issue['product_code']}] Row {issue['row_index']}: '{issue['value']}'")
else:
    print("All Spectro Column B values passed validation.")

TEST BLOCK: TRY APPLYING CORRECTIONS\.

In [ ]:
# ---- Apply confirmed corrections for Spectro Column B anomalies ----
SPECTRO_NAME_CORRECTIONS = {
    "WA15190E": {
        115: "9317AN 249-250 OS",
    }
}

for code, df in spectro_data.items():
    if "lot number_corrected" not in df.columns:
        df["lot number_corrected"] = df["lot number"]

    if code in SPECTRO_NAME_CORRECTIONS:
        for row_idx, corrected_value in SPECTRO_NAME_CORRECTIONS[code].items():
            if row_idx in df.index:
                df.loc[row_idx, "lot number_corrected"] = corrected_value

corrected = spectro_data["WA15190E"].loc[115, ["lot number", "lot number_corrected"]]
print(corrected)

In [ ]:
# ---- Step 3C Block 4: Re-validate after corrections, using lot number_corrected ----

valid_spectro_names_corrected = []
invalid_spectro_names_corrected = []

for code, df in spectro_data.items():
    real_rows = df[df["is_reference_row"] == False]

    for idx, row in real_rows.iterrows():
        value = row["lot number_corrected"]
        value_str = str(value).strip() if not pd.isna(value) else ""

        if not pd.isna(value) and SPECTRO_NAME_PATTERN.match(value_str):
            valid_spectro_names_corrected.append({"product_code": code, "row_index": idx, "value": value_str})
        else:
            invalid_spectro_names_corrected.append({"product_code": code, "row_index": idx, "value": value_str})

print(f"Valid lot number format count (after corrections): {len(valid_spectro_names_corrected)}")

if invalid_spectro_names_corrected:
    print(f"FLAGGED: {len(invalid_spectro_names_corrected)} row(s) still with unexpected format:")
    for issue in invalid_spectro_names_corrected:
        print(f"  [{issue['product_code']}] Row {issue['row_index']}: '{issue['value']}'")
else:
    print("All Spectro Column B values passed validation after corrections.")

## 3D\. Parse Spectro File

### \- 3D Step 1: Cleanse Spectro Columns

In [ ]:
# ---- Step 3D Block 1: Call and display current columns in each Spectro dataframe ----

for code, df in spectro_data.items():
    print(f"\n--- {code} ---")
    print(df.columns.tolist())

In [ ]:
# ---- Rename columns back for clarity ----
for code, df in spectro_data.items():
    df.rename(columns={
        "lot number": "Name",
        "product_code": "Product Code",
        "lot number_corrected": "Name_corrected"
    }, inplace=True)

In [ ]:
# ---- Apply correction directly into Name, drop Name_corrected ----
for code, df in spectro_data.items():
    if "Name_corrected" in df.columns:
        df["Name"] = df["Name_corrected"]
        df.drop(columns=["Name_corrected"], inplace=True)

for code, df in spectro_data.items():
    print(f"\n--- {code} ---")
    print(df.columns.tolist())

### \- 3D Step 2: Splitting values

In [ ]:
for code, df in spectro_data.items():
    real_count = (df["is_reference_row"] == False).sum()
    print(f"{code}: {real_count} row(s) (excluding reference rows)")

In [ ]:
# ---- Block 1: Fill Product Code for non-reference (real) rows ----
for code, df in spectro_data.items():
    df["Product Code"] = code

for code, df in spectro_data.items():
    print(f"\n--- {code} ---")
    print(df[["Name", "Product Code", "is_reference_row"]].to_string(index=False))

In [ ]:
# ---- Block 2: Split Name into lot number + Bag No., reorder columns ----
def split_name_bag(value, is_ref):
    """Returns (lot_number, bag_no_str_or_None, is_oversize_bool). Reference rows untouched.
    Bag No. is always atomic: no "#" prefix, no "OS" text — OS is captured only as the
    separate is_oversize boolean and fully stripped out of the returned bag value."""
    if is_ref:
        return value, None, False

    value = str(value).strip().upper()

    # Parenthesis bag + OS, space optional both sides — e.g. "1234A(1)OS", "1271AL (1) OS"
    m = re.match(rf"^({BASE_LOT})\s?\(({BAG_CORE})\)\s?OS$", value)
    if m:
        return m.group(1), m.group(2), True

    # Parenthesis bag, no OS — e.g. "1234A(1)", "1271AL (1)"
    m = re.match(rf"^({BASE_LOT})\s?\(({BAG_CORE})\)$", value)
    if m:
        return m.group(1), m.group(2), False

    # OS <space> bag, plain or # — e.g. "1234A OS 1", "5409AN OS #19"
    m = re.match(rf"^({BASE_LOT})\s+OS\s+#?({BAG_CORE})$", value)
    if m:
        return m.group(1), m.group(2), True

    # OS#bag, attached — e.g. "5409AN OS#19"
    m = re.match(rf"^({BASE_LOT})\s+OS#({BAG_CORE})$", value)
    if m:
        return m.group(1), m.group(2), True

    # bag <space> OS, plain or # — e.g. "1234A 1 OS", "5409AN #19 OS"
    m = re.match(rf"^({BASE_LOT})\s+#?({BAG_CORE})\s+OS$", value)
    if m:
        return m.group(1), m.group(2), True

    # #bagOS, attached — e.g. "5409AN #19OS"
    m = re.match(rf"^({BASE_LOT})\s+#({BAG_CORE})OS$", value)
    if m:
        return m.group(1), m.group(2), True

    # plain bag, with or without #, no OS — e.g. "1234A 1", "5409AN #19"
    m = re.match(rf"^({BASE_LOT})\s+#?({BAG_CORE})$", value)
    if m:
        return m.group(1), m.group(2), False

    # OS alone, no bag
    m = re.match(rf"^({BASE_LOT})\s+OS$", value)
    if m:
        return m.group(1), None, True

    # lot only
    m = re.match(rf"^({BASE_LOT})$", value)
    if m:
        return m.group(1), None, False

    # Should never hit if 3C validation passed — fallback only.
    return value, None, False


for code, df in spectro_data.items():
    split_results = df.apply(lambda row: split_name_bag(row["Name"], row["is_reference_row"]), axis=1)
    df["Name"] = [r[0] for r in split_results]
    df["Bag No."] = [r[1] for r in split_results]
    df["is_oversize"] = [r[2] for r in split_results]

    # Reorder columns: Color Simulation, Product Code, Name, Bag No., is_oversize, then the rest
    cols = list(df.columns)
    front = ["Color Simulation", "Product Code", "Name", "Bag No.", "is_oversize"]
    remaining = [c for c in cols if c not in front]
    df_cols_ordered = front + remaining
    spectro_data[code] = df[df_cols_ordered]

for code, df in spectro_data.items():
    print(f"\n--- {code} ---")
    print(df[["Color Simulation", "Product Code", "Name", "Bag No.", "is_oversize"]].to_string(index=False))


### \- 3D Step 2 Sub of \(C\): Missing bag correction

In [ ]:
# ---- 3D Step 2 (addendum): Manual correction — add missing Bag No. ----
# Ms. Jam confirmed: these 2 Spectro rows are missing their bag number due to
# operator input omission. Bag range added to match the corresponding QC record.

MANUAL_BAG_NO_CORRECTIONS = {
    ("RA16826E", "3700AO"): "1-2",
    ("WA15190E", "8910AN"): "248-249",
}

for code, df in spectro_data.items():
    for (target_code, target_lot), bag_value in MANUAL_BAG_NO_CORRECTIONS.items():
        if code != target_code:
            continue
        mask = (df["Name"] == target_lot) & (df["Bag No."].isna())
        if mask.any():
            df.loc[mask, "Bag No."] = bag_value
            print(f"{code}: set Bag No.='{bag_value}' for lot {target_lot} ({mask.sum()} row(s))")

### \- 3D Step 3: Check cleaned values

In [ ]:
for code, df in spectro_data.items():
    print(f"\n--- {code} ({len(df)} rows) ---")
    print(df.to_string(index=False))

### \- 3D Step 4: Sort spectro bag column

In [ ]:
def bag_sort_key(bag_val):
    """Returns the starting number of a bag value for sorting (handles ranges and single OS)."""
    if bag_val is None or str(bag_val).strip() == "":
        return -1
    bag_val = str(bag_val).strip()
    if "-" in bag_val:
        try:
            return int(bag_val.split("-")[0])
        except:
            return 9999
    try:
        return int(bag_val)
    except:
        return 9999

for code, df in spectro_data.items():
    real_df = df[df["is_reference_row"] == False].copy()
    real_df["_bag_sort"] = real_df["Bag No."].apply(bag_sort_key)
    real_df_sorted = real_df.sort_values(by=["Name", "_bag_sort"]).drop(columns=["_bag_sort"])

    print(f"\n--- {code} ---")
    print(real_df_sorted[["Name", "Bag No.", "is_oversize"]].to_string(index=False))

### \- 3D Step 5: Check spectro bag column gap

In [ ]:
def parse_bag_range(bag_val):
    if bag_val is None or str(bag_val).strip() == "":
        return None
    bag_val = str(bag_val).strip()
    # Strip an optional parenthesis wrapper (e.g. "(1-40)" -> "1-40").
    if bag_val.startswith("(") and bag_val.endswith(")"):
        bag_val = bag_val[1:-1].strip()
    if "-" in bag_val:
        try:
            a, b = bag_val.split("-")
            return int(a), int(b)
        except:
            return None
    try:
        n = int(bag_val)
        return n, n
    except:
        return None

anomalies = []

for code, df in spectro_data.items():
    real_df = df[df["is_reference_row"] == False]

    for lot, group in real_df.groupby("Name"):
        ranges = []
        for bag_val in group["Bag No."]:
            r = parse_bag_range(bag_val)
            if r:
                ranges.append(r)
        ranges.sort()

        for i in range(1, len(ranges)):
            prev_end = ranges[i-1][1]
            curr_start = ranges[i][0]
            if curr_start <= prev_end:
                anomalies.append({"product_code": code, "lot": lot, "issue": "overlap",
                                   "prev_range": ranges[i-1], "curr_range": ranges[i]})
            elif curr_start > prev_end + 1:
                anomalies.append({"product_code": code, "lot": lot, "issue": "gap",
                                   "prev_range": ranges[i-1], "curr_range": ranges[i]})

if anomalies:
    print(f"FOUND: {len(anomalies)} bag range anomaly(ies)")
    for a in anomalies:
        print(f"  [{a['product_code']}] {a['lot']}: {a['issue']} between {a['prev_range']} and {a['curr_range']}")
else:
    print("No bag range anomalies found.")

TEST BLOCK: TRY APPLYING CORRECTIONS

In [ ]:
# ---- Manual corrections for confirmed typos found in Block 2 ----
# Add entries here only after confirming an anomaly is a typo, not a real gap.
# Format: (product_code, lot_name, original_bag_value) -> corrected_bag_value

BAG_TYPO_CORRECTIONS = {
    ("RA18011E", "7815AN", "27-29"): "27-28",
}

for (code, lot, original_bag), corrected_bag in BAG_TYPO_CORRECTIONS.items():
    df = spectro_data[code]
    mask = (df["Name"] == lot) & (df["Bag No."] == original_bag)
    df.loc[mask, "Bag No."] = corrected_bag

print(f"Applied {len(BAG_TYPO_CORRECTIONS)} typo correction(s) to Spectro Bag No. values.")
for (code, lot, original_bag), corrected_bag in BAG_TYPO_CORRECTIONS.items():
    print(f"  [{code}] {lot}: '{original_bag}' -> '{corrected_bag}'")

In [ ]:
# ---- bag_no_corrected already built in Section A (2H), including remarks fallback.
# No recomputation here — this would overwrite the remarks-recovered values. ----
print(f"bag_no_corrected already set from Section A. Non-blank count: {qc_df['bag_no_corrected'].notna().sum()}")

### \- 3D Step 6: Validate spectro bag number gaps

In [ ]:
# ---- Re-check for bag range anomalies after typo corrections, cross-verify against QC ----

def get_qc_bag_range(product_code, lot_name):
    """Returns a list of individual (lo, hi) bag ranges from QC's bag_no_corrected
    for this product_code + lot, or None. Kept as separate ranges (not collapsed
    into one overall min/max) so gap coverage checks require actual containment
    within a single QC-recorded range, not just falling between the overall span."""
    matches = qc_df[
        (qc_df["product_code_corrected"] == product_code) &
        (qc_df["lot_number_corrected"] == lot_name) &
        (qc_df["bag_no_corrected"].notna())
    ]
    if matches.empty:
        return None

    ranges = []
    for val in matches["bag_no_corrected"]:
        r = parse_bag_range(val)
        if r:
            ranges.append(r)

    if not ranges:
        return None
    return ranges


confirmed_gaps = []

for code, df in spectro_data.items():
    real_df = df[df["is_reference_row"] == False]

    for lot, group in real_df.groupby("Name"):
        ranges = []
        for bag_val in group["Bag No."]:
            r = parse_bag_range(bag_val)
            if r:
                ranges.append(r)
        ranges.sort()

        for i in range(1, len(ranges)):
            prev_end = ranges[i-1][1]
            curr_start = ranges[i][0]

            if curr_start <= prev_end:
                print(f"WARNING: [{code}] {lot} still has an OVERLAP between {ranges[i-1]} and {ranges[i]} — needs manual review, not treated as gap.")
                continue

            if curr_start > prev_end + 1:
                missing_start, missing_end = prev_end + 1, curr_start - 1
                qc_range = get_qc_bag_range(code, lot)

                covers_gap = qc_range and any(
                    r[0] <= missing_start and r[1] >= missing_end for r in qc_range
                )
                if covers_gap:
                    confirmed_gaps.append({
                        "product_code": code, "lot": lot,
                        "missing_start": missing_start, "missing_end": missing_end,
                        "qc_range": qc_range
                    })
                else:
                    print(f"NOTE: [{code}] {lot} has a Spectro-side gap {missing_start}-{missing_end}, "
                          f"but QC does not cover this range (QC range found: {qc_range}) — not auto-filled, needs review.")

if confirmed_gaps:
    print(f"\nCONFIRMED GAPS (QC-verified, safe to fill): {len(confirmed_gaps)}")
    for g in confirmed_gaps:
        print(f"  [{g['product_code']}] {g['lot']}: missing bag(s) {g['missing_start']}-{g['missing_end']} (QC covers {g['qc_range']})")
else:
    print("\nNo confirmed gaps remaining.")

In [ ]:
# ---- Fill confirmed gaps with blank-Spectro-data rows ----

if not confirmed_gaps:
    print("No confirmed gaps to fill — skipping.")
else:
    spectro_only_cols = [c for c in df.columns]  # placeholder, refined per-file below

new_rows_by_code = {}

for gap in confirmed_gaps:
    code = gap["product_code"]
    lot = gap["lot"]
    df = spectro_data[code]

    template_row = df[(df["Name"] == lot) & (df["is_reference_row"] == False)].iloc[0].copy()

    new_row = template_row.copy()
    for col in df.columns:
        if col not in ["Color Simulation", "Product Code", "Name", "Bag No.", "is_oversize", "is_reference_row"]:
            new_row[col] = None
    new_row["Bag No."] = f'{gap["missing_start"]}-{gap["missing_end"]}' if gap["missing_start"] != gap["missing_end"] else str(gap["missing_start"])
    new_row["is_oversize"] = False
    new_row["match_status"] = "QC ONLY (NO SPECTRO READING)"
    new_rows_by_code.setdefault(code, []).append(new_row)

for code, rows in new_rows_by_code.items():
    spectro_data[code] = pd.concat([spectro_data[code], pd.DataFrame(rows)], ignore_index=True)

print(f"Gap-filled {sum(g['missing_end'] - g['missing_start'] + 1 for g in confirmed_gaps)} row(s) across {len(confirmed_gaps)} confirmed gap(s).")

# \(C\) MATCH & MERGE

## 4\. Build flat QC lookup 

In [ ]:
# ---- Step 4: Build one flat QC lookup table ----

# TERMINOLOGY:
#   Mode A — Default matching. Spectro's 'Name' column holds the regular sticker
#            lot number, matched directly against QC's lot_number.
#   Mode B — Special matching. Spectro's 'Name' column holds the INTERNAL lot
#            number instead of the sticker lot number, matched against QC's
#            internal_lot (or the parenthesis-fallback if internal_lot is blank).
#            Applies only to specific clients: PACKAGEWORLD/ABI, ROWELL,
#            DYNAMICCAPS/NICE — see MODE_B_CODES below.

MODE_B_CODES = {"BA12556E", "WA12282E", "WA15151E", "WA15816E",
                "BA17042E", "BA17070E",
                "WA7997E", "WA15218E", "WA15229E"}

QC_COLUMNS_TO_MERGE = ["evaluated_on", "evaluated_by", "customer", "status", "formula_id"]

# OS_REMARKS_PATTERN = re.compile(r"(from\s*os|os)", re.IGNORECASE)
OS_REMARKS_PATTERN = re.compile(r"\b(from\s+os|os)\b", re.IGNORECASE)

def check_os_in_remarks(remarks_value):
    if pd.isna(remarks_value):
        return False
    return bool(OS_REMARKS_PATTERN.search(str(remarks_value)))

def get_internal_lot_key(row):
    """Mode B key: internal_lot column first, then parenthesis-inner fallback."""
    internal_lot = row.get("internal_lot")
    if not pd.isna(internal_lot) and str(internal_lot).strip() != "":
        return str(internal_lot).strip().upper()

    lot_number = str(row["lot_number_corrected"]).strip()
    match = re.match(r"^.*?\(([A-Za-z0-9]+)\)$", lot_number)
    if match:
        inner = match.group(1).strip()
        if re.search(r"[A-Za-z]", inner):
            return inner.upper()
    return None

# ---- Helper: normalize any bag value to canonical 'min-max' or single 'N' form ----
def normalize_bag_range(bag_val):
    if bag_val is None or pd.isna(bag_val) or str(bag_val).strip() == "":
        return None
    bag_val = str(bag_val).strip()
    # Strip an optional parenthesis wrapper (e.g. "(1-40)" -> "1-40") — a valid
    # bag format per BAG_FORMAT_PATTERN that fix_bag_no preserves as-is.
    if bag_val.startswith("(") and bag_val.endswith(")"):
        bag_val = bag_val[1:-1].strip()
    if "-" in bag_val:
        try:
            a, b = bag_val.split("-")
            a, b = int(a), int(b)
            lo, hi = min(a, b), max(a, b)
            return str(lo) if lo == hi else f"{lo}-{hi}"
        except:
            return bag_val
    try:
        return str(int(bag_val))
    except:
        return bag_val

qc_lookup_rows = []

for idx, row in qc_df.iterrows():
    product_code = row["product_code_corrected"]
    is_mode_b = product_code in MODE_B_CODES
    lot_format = row["lot_format"]
    lot_value = str(row["lot_number_corrected"]).strip()

    base_entry = {
        "id": int(row["id"]),
        "product_code": product_code,
        "mode": "B" if is_mode_b else "A",
        "remarks": row.get("remarks"),
    }
    for col in QC_COLUMNS_TO_MERGE:
        base_entry[col] = row.get(col)

    if is_mode_b:
        key = get_internal_lot_key(row)

        # Confirmed pattern: for Mode B, the parenthesis can also hold a plain numeric
        # bag/bag-range (e.g. "3854AL(1-7)") instead of a letter-based internal lot code.
        # get_internal_lot_key already returns None for this case (no letters inside the
        # parens), so detect it here and split into base lot + embedded bag, same as
        # Mode A's bag-in-parens handling — otherwise the whole raw string was wrongly
        # used as the lot key and never matched Spectro's bare lot number.
        embedded_bag_match = None if key else re.match(
            r'^(\d{4}[A-Za-z]{1,2})\((\d{1,3}(?:-\d{1,3})?)\)$', lot_value
        )

        if key:
            lot_key_value = key
            embedded_bag_key = None
        elif embedded_bag_match:
            lot_key_value = embedded_bag_match.group(1)
            embedded_bag_key = normalize_bag_range(embedded_bag_match.group(2))
        else:
            lot_key_value = lot_value.upper()
            embedded_bag_key = None

        bag_key_from_qc = (
            normalize_bag_range(row.get("bag_no_corrected"))
            if pd.notna(row.get("bag_no_corrected")) else None
        )
        # Embedded bag (from the lot number itself) takes priority over bag_no_corrected,
        # since it's the more specific/explicit source for this row.
        resolved_bag_key = embedded_bag_key if embedded_bag_key is not None else bag_key_from_qc

        # Entry 1: bag-less — matches a real physical Spectro reading (no bag suffix),
        # unchanged from before.
        entry_no_bag = dict(base_entry)
        entry_no_bag["lot_key"] = lot_key_value
        entry_no_bag["bag_key"] = None
        qc_lookup_rows.append(entry_no_bag)

        # Entry 2: bag-keyed — matches a Spectro reading with a bag suffix, or a
        # "missing lot" placeholder row (Block 6).
        if resolved_bag_key is not None:
            entry_with_bag = dict(base_entry)
            entry_with_bag["lot_key"] = lot_key_value
            entry_with_bag["bag_key"] = resolved_bag_key
            qc_lookup_rows.append(entry_with_bag)

        continue

    # Mode A
    if lot_format == "single":
        entry = dict(base_entry)
        entry["lot_key"] = lot_value.upper()
        entry["bag_key"] = None
        qc_lookup_rows.append(entry)

    elif lot_format == "range":
        parts = lot_value.split("-")
        if len(parts) == 2:
            expanded = expand_range(parts[0].strip(), parts[1].strip())
            if expanded:
                for lot in expanded:
                    entry = dict(base_entry)
                    entry["lot_key"] = lot.upper()
                    entry["bag_key"] = None
                    qc_lookup_rows.append(entry)

    elif lot_format == "parenthesis":
        # Strip a trailing " OS" (added by normalize_lot_bag_os) before matching,
        # since OS status is tracked separately via remarks and shouldn't block
        # the bag_key parse.
        lot_value_no_os = re.sub(r"\s*OS$", "", lot_value, flags=re.IGNORECASE).strip()

        # bag_single_paren: 1234AB(5)
        m_single = re.match(r"^(\d{4}[A-Za-z]{1,2})\((\d+)\)$", lot_value_no_os)
        # bag_range_paren: 1234AB(5-10) -> expand into individual bags
        m_range = re.match(r"^(\d{4}[A-Za-z]{1,2})\((\d+)-(\d+)\)$", lot_value_no_os)
        # internal_lot_paren handled only under Mode B above; skip here for Mode A

        if m_single:
            entry = dict(base_entry)
            entry["lot_key"] = m_single.group(1).upper()
            entry["bag_key"] = m_single.group(2)
            qc_lookup_rows.append(entry)

        elif m_range:
            base_lot = m_range.group(1).upper()
            bag_start, bag_end = int(m_range.group(2)), int(m_range.group(3))
            entry = dict(base_entry)
            entry["lot_key"] = base_lot
            entry["bag_key"] = normalize_bag_range(f"{bag_start}-{bag_end}")
            qc_lookup_rows.append(entry)

qc_lookup_df = pd.DataFrame(qc_lookup_rows)

print(f"Total QC lookup rows built: {len(qc_lookup_df)}")
print(qc_lookup_df.groupby("mode").size())
# print(qc_lookup_df.head(100).to_string(index=False))

In [ ]:
# ---- Block 6: Add missing Mode B internal lots that have no Spectro reading at all ----
# Mode B lots are sometimes never measured in Spectro at all, since Spectro sampling
# alternates through lots rather than reading every one sequentially. This catches lots
# (including the anchor lot itself, e.g. "8348AM") that exist in QC but have zero rows
# in Spectro, and adds a placeholder row so they still show up in the final report.
#
# NOTE: Intentionally scoped to Mode B only for now — this is the only case raised so far.
# If Ms. Jam wants this same check for Mode A lots later, reuse this block and drop the
# `if code not in MODE_B_CODES: continue` guard below.

missing_lot_rows_by_code = {}

for code, df in spectro_data.items():
    is_mode_b_code = code in MODE_B_CODES  # both modes now processed; branch below

    existing_lots_in_spectro = set(
        df.loc[~df["is_reference_row"], "Name"].astype(str).str.strip().str.upper()
    )

    qc_subset = qc_df[qc_df["product_code_corrected"] == code]
    seen_missing_lots = set()

# ---- Build family ranges from Spectro's own lots (per suffix, clustered by gap <= 2) ----
    def parse_lot(lot):
        m = re.match(r"^(\d+)([A-Za-z]+)$", lot)
        if not m:
            return None
        return int(m.group(1)), m.group(2).upper()

    parsed_spectro = [parse_lot(lot) for lot in existing_lots_in_spectro]
    parsed_spectro = [p for p in parsed_spectro if p]

    by_suffix = {}
    for num, suffix in parsed_spectro:
        by_suffix.setdefault(suffix, []).append(num)

    families = []  # (suffix, min_num, max_num)
    for suffix, nums in by_suffix.items():
        nums = sorted(set(nums))
        cluster = [nums[0]]
        for n in nums[1:]:
            if n - cluster[-1] <= 2:
                cluster.append(n)
            else:
                families.append((suffix, cluster[0], cluster[-1]))
                cluster = [n]
        families.append((suffix, cluster[0], cluster[-1]))

    def within_a_family(num, suffix):
        return any(s == suffix and lo <= num <= hi for s, lo, hi in families)

    for _, qc_row in qc_subset.iterrows():
        if is_mode_b_code:
            key = get_internal_lot_key(qc_row)
            if key:
                lot_values = [key]  # child lot — from internal_lot column OR parenthesis extraction
            else:
                lot_values = [str(qc_row["lot_number_corrected"]).strip().upper()]  # anchor lot
        else:
            # Mode A: expand this QC row into the individual lot(s) it covers,
            # same expansion rules used when building qc_lookup_df in Step 4.
            lot_format = qc_row.get("lot_format")
            raw_lot = str(qc_row["lot_number_corrected"]).strip()

            if lot_format == "single":
                lot_values = [raw_lot.upper()]

            elif lot_format == "range":
                parts = raw_lot.split("-")
                if len(parts) == 2:
                    expanded = expand_range(parts[0].strip(), parts[1].strip())
                    lot_values = [l.upper() for l in expanded] if expanded else []
                else:
                    lot_values = []

            elif lot_format == "parenthesis":
                m_single = re.match(r"^(\d{4}[A-Za-z]{1,2})\((\d+)\)$", raw_lot)
                m_range = re.match(r"^(\d{4}[A-Za-z]{1,2})\((\d+)-(\d+)\)$", raw_lot)
                if m_single:
                    lot_values = [m_single.group(1).upper()]
                elif m_range:
                    lot_values = [m_range.group(1).upper()]
                else:
                    lot_values = []
            else:
                lot_values = []

        for lot_value in lot_values:
            if lot_value in existing_lots_in_spectro or lot_value in seen_missing_lots:
                continue

            parsed = parse_lot(lot_value)
            if not parsed or not within_a_family(*parsed):
                continue  # outside every Spectro family's range — not needed this batch

            seen_missing_lots.add(lot_value)

            new_row = df.iloc[0].copy()
            for col in df.columns:
                new_row[col] = None
            new_row["Product Code"] = code
            new_row["Name"] = lot_value
            new_row["Bag No."] = qc_row.get("bag_no_corrected")   # from QC's own bag_no column
            # new_row["is_oversize"] = False
            new_row["is_oversize"] = check_os_in_remarks(qc_row.get("remarks"))
            new_row["is_reference_row"] = False
            new_row["match_status"] = "QC ONLY (NO SPECTRO READING)"

            missing_lot_rows_by_code.setdefault(code, []).append(new_row)

for code, rows in missing_lot_rows_by_code.items():
    spectro_data[code] = pd.concat([spectro_data[code], pd.DataFrame(rows)], ignore_index=True)

total_added = sum(len(rows) for rows in missing_lot_rows_by_code.values())
print(f"Added {total_added} missing-lot placeholder row(s) across {len(missing_lot_rows_by_code)} Mode B file(s).")
for code, rows in missing_lot_rows_by_code.items():
    for r in rows:
        print(f"  [{code}] {r['Name']} — Bag No. {r['Bag No.']}")

## 4A\. Restrict bag expansion to Mode A only

In [ ]:
# ---- Step 4 (addendum) FIX: only expand bag_no_corrected for Mode A entries ----

additional_lookup_rows = []
rows_to_remove_indices = []

for i, entry in enumerate(qc_lookup_rows):
    if entry["mode"] != "A":   # <-- the fix: skip Mode B entirely
        continue

    qc_row_id = entry["id"]
    matching_qc_row = qc_df[qc_df["id"] == qc_row_id]
    if matching_qc_row.empty:
        continue
    qc_row = matching_qc_row.iloc[0]

    lot_format = qc_row["lot_format"]
    bag_no_val = qc_row.get("bag_no_corrected")

    if entry["bag_key"] is None and lot_format in ("single", "range") \
            and bag_no_val is not None and str(bag_no_val).strip() != "":

        new_entry = dict(entry)
        new_entry["bag_key"] = normalize_bag_range(bag_no_val)
        additional_lookup_rows.append(new_entry)

        rows_to_remove_indices.append(i)

qc_lookup_rows = [e for i, e in enumerate(qc_lookup_rows) if i not in rows_to_remove_indices]
qc_lookup_rows.extend(additional_lookup_rows)

qc_lookup_df = pd.DataFrame(qc_lookup_rows)

print(f"Additional bag-expanded entries added: {len(additional_lookup_rows)}")
print(f"Total QC lookup rows now: {len(qc_lookup_df)}")
print(qc_lookup_df.groupby("mode").size())

## 4B\. Clean NONE/NAN artifacts from lookup keys

In [ ]:
# ---- Fix: Clean qc_lookup_df's bag_key and lot_key to remove text "NONE"/"NAN" artifacts ----
qc_lookup_df["bag_key"] = qc_lookup_df["bag_key"].replace({"NONE": None, "NAN": None, "nan": None})
qc_lookup_df["lot_key"] = qc_lookup_df["lot_key"].replace({"NONE": None, "NAN": None, "nan": None})

print(qc_lookup_df["bag_key"].isna().sum(), "bag_key entries are now real None")

## 4C\. Match Spectro rows against QC lookup

In [ ]:
# ---- Step 4A: Matching loop ----
# Matches each real Spectro row against qc_lookup_df using (product_code, lot_key, bag_key, mode).
# Reference rows are skipped from matching entirely.
# Oversize cross-check: if Spectro row is flagged is_oversize=True, check the matched
# QC row's remarks for "OS" or "from OS" (any case). This does not affect the match itself.

#OS_REMARKS_PATTERN = re.compile(r"(from\s*os|os)", re.IGNORECASE)

#def check_os_in_remarks(remarks_value):
#    if pd.isna(remarks_value):
#        return False
#    return bool(OS_REMARKS_PATTERN.search(str(remarks_value)))


all_matched_rows = []

for code, df in spectro_data.items():
    is_mode_b = code in MODE_B_CODES
    mode_label = "B" if is_mode_b else "A"

    lookup_subset = qc_lookup_df[
        (qc_lookup_df["product_code"] == code) & (qc_lookup_df["mode"] == mode_label)
    ]

    matched = unmatched = skipped_ref = 0

    for idx, row in df.iterrows():
        is_gap_filled = row.get("match_status") == "QC ONLY (NO SPECTRO READING)"

        result_row = row.copy()
        for col in QC_COLUMNS_TO_MERGE:
            result_row[col] = None
        result_row["match_status"] = None
        result_row["Oversize"] = bool(row.get("is_oversize", False))
        result_row["os_remarks_match"] = None

        if row["is_reference_row"]:
            result_row["match_status"] = "REFERENCE ROW"
            skipped_ref += 1
            all_matched_rows.append(result_row)
            continue

        lot_key = str(row["Name"]).strip().upper()
        spectro_bag = normalize_bag_range(row["Bag No."])
        spectro_range = parse_bag_range(spectro_bag) if spectro_bag else None  # (lo, hi) or None

        candidates = lookup_subset[lookup_subset["lot_key"] == lot_key]

        if spectro_range is not None:
            def qc_covers_spectro(qc_bag_val):
                qc_range = parse_bag_range(qc_bag_val)
                if qc_range is None:
                    return False
                return qc_range[0] <= spectro_range[0] and qc_range[1] >= spectro_range[1]

            candidates = candidates[candidates["bag_key"].apply(qc_covers_spectro)]
        else:
            candidates = candidates[candidates["bag_key"].isna()]

        if candidates.empty:
            result_row["match_status"] = "NO MATCH FOUND"
            unmatched += 1
            all_matched_rows.append(result_row)
            continue

        if len(candidates) > 1:
            candidates = candidates.copy()
            candidates["evaluated_on_dt"] = pd.to_datetime(candidates["evaluated_on"], errors="coerce")
            candidates = candidates.sort_values("evaluated_on_dt", ascending=False)
            best_match = candidates.iloc[0]
            result_row["match_status"] = f"MATCHED (latest of {len(candidates)})"
        else:
            best_match = candidates.iloc[0]
            result_row["match_status"] = "MATCHED"

        if is_gap_filled:
            result_row["match_status"] = "QC ONLY (NO SPECTRO READING)"

        for col in QC_COLUMNS_TO_MERGE:
            result_row[col] = best_match[col]

        if result_row["Oversize"]:
            result_row["os_remarks_match"] = check_os_in_remarks(best_match["remarks"])

        matched += 1
        all_matched_rows.append(result_row)

    print(f"{code} (Mode {mode_label}): Matched={matched}, Unmatched={unmatched}, Reference={skipped_ref}")

final_df = pd.DataFrame(all_matched_rows)
print(f"\nTotal rows: {len(final_df)}")
print(final_df["match_status"].value_counts())

In [ ]:
# ---- Step 4A (diagnostic): List all NO MATCH FOUND rows ----

no_match_rows = final_df[final_df["match_status"] == "NO MATCH FOUND"]

print(f"Total NO MATCH FOUND rows: {len(no_match_rows)}")
print(no_match_rows[["Product Code", "Name", "Bag No.", "is_oversize"]].to_string(index=False))

# \(D\) OUTPUT

## 5\. Rename columns

In [ ]:
final_df = final_df.rename(columns={
    "Name": "Lot Number",
    "Bag No.": "Bag Number",
    "Judgement": "Spectro Judgement",
    "status": "Final QC Status",
    "evaluated_on": "Evaluated On",
    "evaluated_by": "Evaluated By",
    "customer": "Customer",
    "formula_id": "Formula ID",
    "match_status": "Match Status",
})

## 5A\. Clean value formats

In [ ]:
def clean_upper_nospace(value):
    """Uppercase, strip all spaces. Leaves real blanks untouched."""
    if pd.isna(value) or str(value).strip() == "":
        return value
    return str(value).strip().upper().replace(" ", "")

final_df["Evaluated By"] = final_df["Evaluated By"].apply(clean_upper_nospace)
final_df["Customer"] = final_df["Customer"].apply(clean_upper_nospace)

# Oversize -> real boolean
final_df["Oversize"] = final_df["Oversize"].astype(bool)

# Formula ID -> integer, not float (fixes 17026.0 issue). Int64 (capital I)
# keeps it a nullable-int type so blanks stay blank instead of forcing 0.
final_df["Formula ID"] = pd.to_numeric(final_df["Formula ID"], errors="coerce").astype("Int64")

## 5B\. Column order

In [ ]:
# ---- Define final output column order ----

FINAL_COLUMN_ORDER = [
    "Color Simulation", "Lot Number", "Product Code", "Bag Number", "Oversize",
    "Date Time", "ΔE*00", "L*", "C*", "h°", "a*", "b*",
    "ΔL*", "ΔC*", "ΔH*", "Δa*", "Δb*", "Color Offset",
    "Spectro Judgement", "Final QC Status", "Evaluated On", "Evaluated By",
    "Customer", "Formula ID", "Match Status",
]

final_df = final_df[FINAL_COLUMN_ORDER]
print(final_df.columns.tolist())
print(final_df.head(10))

## 5C: Sort: reference rows first per product code group

In [ ]:
# ---- Step 5C: Sort — reference rows first, then by lot number, then bag ascending ----

final_df["_ref_sort"] = (final_df["Match Status"] != "REFERENCE ROW").astype(int)
# 0 = reference row (sorts first), 1 = everything else

def bag_sort_key(bag_val):
    """Numeric sort key for Bag Number. Ranges like '1-2' sort by their start number."""
    if pd.isna(bag_val) or str(bag_val).strip() == "":
        return -1  # blanks (e.g. reference rows) sort first
    bag_str = str(bag_val).strip()
    if "-" in bag_str:
        try:
            return int(bag_str.split("-")[0])
        except:
            return 0
    try:
        return int(bag_str)
    except:
        return 0

final_df["_bag_sort"] = final_df["Bag Number"].apply(bag_sort_key)

final_df = final_df.sort_values(
    by=["Product Code", "_ref_sort", "Lot Number", "_bag_sort"],
    kind="stable"
).drop(columns=["_ref_sort", "_bag_sort"]).reset_index(drop=True)

## 5D\. Gray formatting for reference rows

In [ ]:
HEADER_FILL = PatternFill(start_color="2F4F4F", end_color="2F4F4F", fill_type="solid")
HEADER_FONT = Font(color="FFFFFF", size=12, bold=True)

FAIL_FILL = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")  # burnt red-orange, per screenshot
FAIL_FONT = Font(color="FFFFFF", bold=True)  # white text
REF_FILL = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
HEADER_ALIGNMENT = Alignment(horizontal="center", vertical="center", wrap_text=False)
NO_MATCH_FONT = Font(color="FF0000", bold=True)
NO_MATCH_COLUMNS = {"Lot Number", "Product Code", "Bag Number"}

def apply_output_formatting(ws, df):
    col_name_to_idx = {name: idx for idx, name in enumerate(df.columns, start=1)}

    # 1. Header row styling
    for col_idx in range(1, len(df.columns) + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = HEADER_ALIGNMENT

    # 2 & 3. Per-row styling (data rows start at Excel row 2)
    for row_idx, row in df.iterrows():
        excel_row = row_idx + 2
        is_ref = row["Match Status"] == "REFERENCE ROW"
        is_fail = str(row["Final QC Status"]).strip().upper() == "FAILED"
        is_no_match = row["Match Status"] == "NO MATCH FOUND"

        for col_idx in range(1, len(df.columns) + 1):
            cell = ws.cell(row=excel_row, column=col_idx)

            if is_fail:
                cell.fill = FAIL_FILL
                cell.font = FAIL_FONT
            elif is_ref:
                cell.fill = REF_FILL
                cell.font = Font(bold=True)

        # 4. Genuine no-match: red text on Lot Number, Product Code, Bag Number only
        if is_no_match:
            for col_name in NO_MATCH_COLUMNS:
                ws.cell(row=excel_row, column=col_name_to_idx[col_name]).font = NO_MATCH_FONT

## 5E\. Write final Excel file

In [ ]:
ph_now = datetime.now(ZoneInfo("Asia/Manila"))

ph_now = datetime.now(ZoneInfo("Asia/Manila"))

VERSION_COUNTER_FILE = "output/.version_counter"
os.makedirs("output", exist_ok=True)
if os.path.exists(VERSION_COUNTER_FILE):
    with open(VERSION_COUNTER_FILE, "r") as f:
        VERSION_LABEL = int(f.read().strip()) + 1
else:
    VERSION_LABEL = 1
with open(VERSION_COUNTER_FILE, "w") as f:
    f.write(str(VERSION_LABEL))

OUTPUT_PATH = f"output/v.0.{VERSION_LABEL} - Final Merge - {ph_now.strftime('%B %d, %Y - %H_%M')}.xlsx"

ref_mask = final_df["Match Status"] == "REFERENCE ROW"
qc_only_mask = final_df["Match Status"] == "QC ONLY (NO SPECTRO READING)"

sheets = {
    "RAW_DATA": final_df,
    "COMPLETE_DATA": final_df[~ref_mask & ~qc_only_mask].reset_index(drop=True),
    "REFERENCE_DATA": final_df[ref_mask].reset_index(drop=True),
    "INCOMPLETE_DATA": final_df[qc_only_mask].reset_index(drop=True),
}

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        df.to_excel(writer, index=False, sheet_name=sheet_name)
        ws = writer.sheets[sheet_name]
        apply_output_formatting(ws, df)

        for col_idx, col_name in enumerate(df.columns, start=1):
            max_len = max(
                df[col_name].astype(str).map(len).max() if len(df) else 0,
                len(str(col_name))
            )
            ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = max_len + 4

        ws.row_dimensions[1].height = 22

        # Filtered/ready view: enable AutoFilter across the header row's full range
        if len(df):
            last_col_letter = ws.cell(row=1, column=len(df.columns)).column_letter
            ws.auto_filter.ref = f"A1:{last_col_letter}{ws.max_row}"

        # Freeze the header row so it stays visible while scrolling
        ws.freeze_panes = "A2"

print(f"Saved: {OUTPUT_PATH}")
print(f"Total rows: {len(final_df)}")
for sheet_name, df in sheets.items():
    print(f"  {sheet_name}: {len(df)} rows")

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=3b53bd28-7e59-47fa-80c2-b8dc99257c16' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>